# 13.6 - vLLM & GKE Production

**Module 13 - Cost & Performance** | Netsetos GenAI Engineering

This notebook works through vLLM & GKE Production hands-on: Why vLLM: continuous batching; PagedAttention: the KV cache, paged; The batching knob: throughput vs latency; The vLLM server + tensor parallelism; Deploy on GKE; Autoscale on the right signal.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Every block in this notebook is a tiny arithmetic model of a vLLM/GKE serving decision - no GPU, no API, no network. This first cell just notes the (commented-out) install and the fact that the numbers are seeded and illustrative, so you can run the whole notebook on a laptop.

In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install openai -q

# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A comment-only setup cell. There is no live install and no import to run - the later blocks use only Python built-ins (`math` is imported inline where it is needed). It exists to flag that `pip install openai` is optional and that every figure downstream is a deliberately chosen illustrative constant, not a live measurement.

**How the code works - step by step**
- **Commented `pip install openai`** - only needed if you actually want to call an OpenAI-compatible endpoint; the models here never do.
- **Reproducibility note** - a reminder that the lesson's numbers are fixed/seeded so your output matches the walkthroughs.

**In one line:** environment prep only - nothing executes.

**What you'll see:** (no output - environment prep)

## 1 - Why vLLM: continuous batching

Static batching launches a fixed group of requests together and waits for the **longest** one to finish, so short requests hold their GPU slot idle. Continuous batching (vLLM's scheduler) refills a finished slot from the queue on every forward pass, so the GPU never idles. This block prices the wasted GPU time on one batch to show why that single change is the biggest win in serving.

In [ ]:
# STATIC batching launches a fixed batch together and waits for the LONGEST to finish - the short
# requests hold their GPU slot idle. CONTINUOUS batching (vLLM) reschedules EVERY forward pass: a finished
# request is evicted immediately and a queued one takes its slot, so the GPU never idles. (Illustrative.)
outputs = [64, 96, 128, 512]           # output tokens each of 4 batched requests generates
batch, longest = len(outputs), max(outputs)
static_slotsteps = batch * longest     # static reserves batch x longest slot-steps (all wait for the longest)
useful = sum(outputs)                  # the tokens actually generated
static_waste = 1 - useful / static_slotsteps
print("A batch of {} requests generating {} tokens (longest {}):".format(batch, outputs, longest))
print("  STATIC: reserves {} slot-steps (all {} slots held until the {}-token request finishes)".format(static_slotsteps, batch, longest))
print("  useful work: {} tokens -> {:.0%} of the GPU-time is WASTED padding (idle slots).".format(useful, static_waste))
print()
print("  CONTINUOUS (vLLM): a finished slot is refilled from the queue the same step, so slots stay busy.")
print("  With a full queue, that recovers the wasted {:.0%} - about {:.1f}x the throughput on this batch,".format(static_waste, static_slotsteps / useful))
print("  and on real deep-queue workloads continuous batching reaches 10-23x while ALSO cutting p50 latency.")
print("Continuous batching is the single biggest serving win - it is why vLLM exists.")

# Output:
# A batch of 4 requests generating [64, 96, 128, 512] tokens (longest 512):
#   STATIC: reserves 2048 slot-steps (all 4 slots held until the 512-token request finishes)
#   useful work: 800 tokens -> 61% of the GPU-time is WASTED padding (idle slots).
#
#   CONTINUOUS (vLLM): a finished slot is refilled from the queue the same step, so slots stay busy.
#   With a full queue, that recovers the wasted 61% - about 2.6x the throughput on this batch,
#   and on real deep-queue workloads continuous batching reaches 10-23x while ALSO cutting p50 latency.
# Continuous batching is the single biggest serving win - it is why vLLM exists.

**Explanation**

A back-of-envelope waste calculator, not a model call. It takes four request lengths, computes how many slot-steps static batching burns (batch size times the longest request) versus the tokens actually generated, and turns the gap into a waste percentage and a throughput multiplier. Read it as: static reserves for the worst case, continuous recovers the difference.

**How the code works - step by step**
- **`outputs` / `batch` / `longest`** - four requests generating 64, 96, 128, 512 tokens; batch size 4, longest 512.
- **`static_slotsteps = batch * longest`** - static batching holds all 4 slots until the 512-token request finishes = 2048 slot-steps.
- **`useful = sum(outputs)`** - the tokens actually produced = 800.
- **`static_waste = 1 - useful / static_slotsteps`** - the fraction of GPU-time spent on idle padding.
- **Throughput note** - `static_slotsteps / useful` is the speed-up continuous batching recovers on this batch; real deep-queue traffic reaches 10-23x.

**In one line:** reserve for the longest -> most of the GPU sits idle -> refilling from the queue recovers it.

**What you'll see:** Prints that the 4-request batch reserves 2048 slot-steps for 800 useful tokens, so 61% of GPU-time is wasted padding; continuous batching recovers that (~2.6x here, 10-23x on real workloads) while cutting p50 latency.

## 2 - PagedAttention: the KV cache, paged

Continuous batching only helps if enough sequences fit on the card, and the KV cache is what limits that. Naive serving reserves one contiguous block of the full `max_model_len` per request; PagedAttention stores the cache in fixed 16-token pages (like OS virtual memory), so waste is at most one partial block. This block compares the VRAM reserved under each scheme.

In [ ]:
# PagedAttention: the KV cache is the other bottleneck. Naive serving reserves a CONTIGUOUS block of
# max_model_len per request (for tokens it may never generate) -> huge waste + fragmentation. PagedAttention
# stores the KV cache in fixed BLOCKS (like OS virtual-memory pages), so waste is at most one block.
max_len, block = 4096, 16               # max_model_len tokens; a KV page is 16 tokens
actual = [200, 512, 1500]               # the actual context lengths of 3 live requests
used = sum(actual)
naive_reserved = len(actual) * max_len                                  # each reserves the full max_len
paged_reserved = sum((a + block - 1) // block * block for a in actual)  # each reserves ceil(len/block) pages
print("Three requests with contexts {} (max_model_len {}):".format(actual, max_len))
print("  naive (contiguous max reserve): {:>6} tokens reserved, {} used -> {:.0%} wasted".format(naive_reserved, used, 1 - used / naive_reserved))
print("  PagedAttention (16-token pages): {:>6} tokens reserved, {} used -> {:.0%} wasted".format(paged_reserved, used, 1 - used / paged_reserved))
print()
seqs_naive = naive_reserved // max_len          # in a fixed KV budget, naive fits few (each hogs max_len)
mult = naive_reserved / paged_reserved
print("In the SAME VRAM, PagedAttention fits about {:.1f}x more concurrent sequences (it packs the KV cache tight).".format(mult))
print("More concurrent sequences = a fuller batch = more of the throughput continuous batching can deliver.")
print("When the KV cache fills, vLLM PREEMPTS (evicts) an older sequence to make room - watch gpu_cache_usage_perc.")

# Output:
# Three requests with contexts [200, 512, 1500] (max_model_len 4096):
#   naive (contiguous max reserve):  12288 tokens reserved, 2212 used -> 82% wasted
#   PagedAttention (16-token pages):   2224 tokens reserved, 2212 used -> 1% wasted
#
# In the SAME VRAM, PagedAttention fits about 5.5x more concurrent sequences (it packs the KV cache tight).
# More concurrent sequences = a fuller batch = more of the throughput continuous batching can deliver.
# When the KV cache fills, vLLM PREEMPTS (evicts) an older sequence to make room - watch gpu_cache_usage_perc.

**Explanation**

A memory-waste model that mirrors the batching model, but for VRAM instead of GPU-time. It reserves the full max length per request the naive way, versus rounding each request up to whole 16-token pages the paged way, and divides to show how many more sequences fit in the same VRAM. The core idea: pay for pages you use, not for a max-length reservation you mostly don't.

**How the code works - step by step**
- **`max_len, block = 4096, 16`** - the context cap and the KV page size in tokens.
- **`actual = [200, 512, 1500]`** - three live requests' real context lengths; `used` sums them.
- **`naive_reserved = len(actual) * max_len`** - each request reserves the full 4096, regardless of what it uses.
- **`paged_reserved = sum(ceil(a/block)*block ...)`** - each request reserves only whole pages covering its actual length.
- **`mult = naive_reserved / paged_reserved`** - how many more concurrent sequences fit in the same VRAM.
- **Preemption note** - when the cache fills, vLLM evicts an older sequence; watch `gpu_cache_usage_perc`.

**In one line:** page the KV cache instead of reserving the max -> waste collapses -> several times more sequences fit.

**What you'll see:** Prints that naive reserves 12288 tokens (82% wasted) versus PagedAttention's 2224 tokens (1% wasted) for the same 2212 used, fitting ~5.5x more concurrent sequences, and notes vLLM preempts an older sequence when the cache fills.

## 3 - The batching knob: throughput vs latency

`--max-num-seqs` sets how many sequences run at once. A bigger batch pushes aggregate throughput toward the GPU ceiling, but the sequences share one GPU so each request's own token rate falls. This block sweeps the knob to find the knee - the batch size where aggregate throughput saturates and past which you only add latency.

In [ ]:
# The knob: --max-num-seqs sets how many sequences run at once. A bigger batch pushes AGGREGATE throughput
# toward the GPU ceiling, but the sequences share the GPU, so each request's own token rate FALLS. Find the knee.
GPU_CEILING = 2400          # aggregate decode tokens/s at saturation (illustrative, 70B FP8 on one H100)
PER_SEQ_MAX = 90            # a single lightly-loaded sequence decodes ~90 tok/s
print("max-num-seqs  aggregate tok/s   per-request tok/s   note")
for m in [8, 32, 128, 512]:
    agg = min(GPU_CEILING, m * PER_SEQ_MAX)     # throughput rises with batch until it hits the ceiling
    per_req = agg / m                            # but each of m sequences gets a share
    note = "under-utilised" if agg < GPU_CEILING else "saturated (the knee is ~here)"
    print("  {:>4}          {:>6}            {:>5.0f}              {}".format(m, agg, per_req, note))
print()
print("Small batch: low throughput, snappy per-request. Big batch: full GPU, slower per-request. The knee is")
print("where aggregate throughput saturates ({} tok/s here) - past it you add latency for almost no throughput.".format(GPU_CEILING))
print("Chunked prefill (default 2048-token chunks) interleaves a long prompt's prefill with ongoing decodes,")
print("so one big prompt does not block everyone else (head-of-line blocking). Set max-num-seqs to your traffic.")

# Output:
# max-num-seqs  aggregate tok/s   per-request tok/s   note
#      8             720               90              under-utilised
#     32            2400               75              saturated (the knee is ~here)
#    128            2400               19              saturated (the knee is ~here)
#    512            2400                5              saturated (the knee is ~here)
#
# Small batch: low throughput, snappy per-request. Big batch: full GPU, slower per-request. The knee is
# where aggregate throughput saturates (2400 tok/s here) - past it you add latency for almost no throughput.
# Chunked prefill (default 2048-token chunks) interleaves a long prompt's prefill with ongoing decodes,
# so one big prompt does not block everyone else (head-of-line blocking). Set max-num-seqs to your traffic.

**Explanation**

A throughput/latency sweep, not a benchmark - it models the GPU as a hard aggregate ceiling and each sequence as an equal share of whatever the batch produces. As the batch grows, aggregate rises until it hits the ceiling and then flattens, while per-request rate keeps dropping. The knee is where the aggregate first saturates.

**How the code works - step by step**
- **`GPU_CEILING = 2400`** - the aggregate decode tokens/s the GPU can sustain at saturation.
- **`PER_SEQ_MAX = 90`** - what one lightly-loaded sequence decodes on its own.
- **loop over `[8, 32, 128, 512]`** - for each batch size `m`:
  - **`agg = min(GPU_CEILING, m * PER_SEQ_MAX)`** - throughput climbs with the batch until it hits the ceiling.
  - **`per_req = agg / m`** - each of `m` sequences gets an equal share, so per-request rate falls.
  - **`note`** - labels whether the GPU is under-utilised or saturated (the knee).
- **Chunked-prefill note** - a long prompt's prefill is split into ~2048-token chunks interleaved with decodes so it doesn't block everyone (head-of-line blocking).

**In one line:** bigger batch -> aggregate saturates at the knee while per-request rate keeps falling.

**What you'll see:** Prints a table showing throughput rising 720 -> 2400 tok/s (saturating by max-num-seqs 32, the knee) while per-request rate falls 90 -> 5 tok/s, plus the chunked-prefill note about not blocking the batch.

## 4 - The vLLM server + tensor parallelism

`vllm serve <model>` starts an OpenAI-compatible server on port 8000 - the same `/v1/chat/completions` your app already calls. When a model is too big for one GPU, `--tensor-parallel-size N` shards every layer across N GPUs. This block computes the right N from the model size and one GPU's usable VRAM.

In [ ]:
import math
# A model too big for one GPU is split across GPUs with TENSOR PARALLELISM (--tensor-parallel-size). The
# rule of thumb: TP >= ceil(model VRAM / usable GPU VRAM), rounded up to a power of 2 (it must divide the heads).
GPU_VRAM, UTIL = 80, 0.90        # an 80 GB GPU; --gpu-memory-utilization 0.90 leaves room for the KV cache
usable = GPU_VRAM * UTIL
def tp_size(model_gb):
    need = math.ceil(model_gb / usable)
    return 1 if need <= 1 else 2 ** math.ceil(math.log2(need))
for name, gb in [("70B FP8 (~70 GB)", 70), ("70B FP16 (~140 GB)", 140), ("405B FP8 (~405 GB)", 405)]:
    tp = tp_size(gb)
    print("  {:<20} needs tensor-parallel-size {}  ({} x {:.0f} GB usable = {:.0f} GB across GPUs)".format(name, tp, tp, usable, tp * usable))
print()
print("`vllm serve <model> --tensor-parallel-size {} --gpu-memory-utilization {} --max-model-len 8192`".format(tp_size(140), UTIL))
print("gives an OpenAI-compatible endpoint on :8000 - the SAME /v1/chat/completions your app already calls.")
print("TP splits every layer across the GPUs; keep it inside one node (fast NVLink) and use pipeline parallel across nodes.")

# Output:
#   70B FP8 (~70 GB)     needs tensor-parallel-size 1  (1 x 72 GB usable = 72 GB across GPUs)
#   70B FP16 (~140 GB)   needs tensor-parallel-size 2  (2 x 72 GB usable = 144 GB across GPUs)
#   405B FP8 (~405 GB)   needs tensor-parallel-size 8  (8 x 72 GB usable = 576 GB across GPUs)
#
# `vllm serve <model> --tensor-parallel-size 2 --gpu-memory-utilization 0.9 --max-model-len 8192`
# gives an OpenAI-compatible endpoint on :8000 - the SAME /v1/chat/completions your app already calls.
# TP splits every layer across the GPUs; keep it inside one node (fast NVLink) and use pipeline parallel across nodes.

**Explanation**

A GPU-sizing calculator. It takes one GPU's usable VRAM (physical size times `--gpu-memory-utilization`) and, for each model size, computes the smallest power-of-two tensor-parallel size that fits, because the shard count must divide the attention heads evenly. Read it as a lookup: model VRAM over usable GPU VRAM, rounded up to a power of two.

**How the code works - step by step**
- **`GPU_VRAM, UTIL = 80, 0.90`** - an 80 GB card; 0.90 leaves headroom for the KV cache, giving `usable = 72 GB`.
- **`tp_size(model_gb)`** - `ceil(model_gb / usable)`, then rounded up to the next power of two (1 if it fits on one card).
- **loop over three models** - 70B FP8 (~70 GB), 70B FP16 (~140 GB), 405B FP8 (~405 GB) - prints the required tensor-parallel size and the total VRAM across GPUs.
- **`vllm serve ...` line** - the assembled command with `--tensor-parallel-size`, `--gpu-memory-utilization`, `--max-model-len`, exposing the OpenAI-compatible endpoint on :8000.
- **Placement note** - keep tensor parallelism inside one node (fast NVLink); use pipeline parallelism across nodes.

**In one line:** model VRAM / usable GPU VRAM, rounded to a power of two = your tensor-parallel size.

**What you'll see:** Prints that 70B FP8 needs tensor-parallel-size 1, 70B FP16 needs 2, and 405B FP8 needs 8, then shows the `vllm serve` command and the note that TP shards every layer and should stay inside one NVLink node.

## 5 - Deploy on GKE

A GKE Deployment runs N vLLM pods, each requesting a GPU on a GPU node pool. The catch is the cold start: a pod isn't ready until it has downloaded and loaded the weights - minutes, not seconds. This block models that cold start so you see why you keep min replicas warm and cache the weights on a volume.

In [ ]:
# On GKE, a Deployment runs N vLLM pods, each claiming a GPU (resources: nvidia.com/gpu: 1). The catch is
# COLD START: a pod is not ready until it has DOWNLOADED and loaded the weights - minutes, not seconds.
replicas, gpu_per_pod = 3, 1
model_gb, net_gbps, load_s = 140, 2.5, 60          # weights size; node download bandwidth; weight-load time
download_s = model_gb * 8 / net_gbps               # GB -> gigabits / (gigabits per second)
cold_start_s = download_s + load_s
print("Deployment: {} replicas x {} GPU = {} GPUs (each pod requests nvidia.com/gpu: {}).".format(replicas, gpu_per_pod, replicas * gpu_per_pod, gpu_per_pod))
print("Cold start of one pod (a {} GB model):".format(model_gb))
print("  download: {:.0f} s + load: {} s = {:.0f} s (~{:.0f} min) before the readiness probe passes.".format(download_s, load_s, cold_start_s, cold_start_s / 60))
print()
print("So a burst cannot be met by a cold pod in time. Two fixes:")
print("  1) mount a PersistentVolumeClaim at the HF cache -> the weights are already there, skip the download.")
print("  2) keep min replicas warm (never scale to zero) so there is always capacity while a new pod boots.")
print("Readiness/liveness probes hit /health; a pod only takes traffic AFTER the model is loaded.")

# Output:
# Deployment: 3 replicas x 1 GPU = 3 GPUs (each pod requests nvidia.com/gpu: 1).
# Cold start of one pod (a 140 GB model):
#   download: 448 s + load: 60 s = 508 s (~8 min) before the readiness probe passes.
#
# So a burst cannot be met by a cold pod in time. Two fixes:
#   1) mount a PersistentVolumeClaim at the HF cache -> the weights are already there, skip the download.
#   2) keep min replicas warm (never scale to zero) so there is always capacity while a new pod boots.
# Readiness/liveness probes hit /health; a pod only takes traffic AFTER the model is loaded.

**Explanation**

A cold-start timing model. It converts a model's size into a download time over the node's bandwidth (GB to gigabits, divided by gigabits per second), adds a fixed weight-load time, and reports the total before a readiness probe can pass. The point is that a fresh pod cannot absorb a sudden burst, which motivates the two fixes.

**How the code works - step by step**
- **`replicas, gpu_per_pod = 3, 1`** - the Deployment's shape; each pod requests `nvidia.com/gpu: 1`.
- **`model_gb, net_gbps, load_s = 140, 2.5, 60`** - weights size, download bandwidth, and weight-load time.
- **`download_s = model_gb * 8 / net_gbps`** - GB to gigabits over the link speed.
- **`cold_start_s = download_s + load_s`** - total time before the readiness probe passes.
- **Fix 1** - mount a PersistentVolumeClaim at the HF cache so weights are already on disk (skip the download).
- **Fix 2** - keep min replicas warm (never scale a latency service to zero) so there is capacity while a new pod boots.
- **Probe note** - `/health` readiness/liveness probes gate traffic until the model is loaded.

**In one line:** download + load = minutes -> keep warm replicas and cache weights so a burst isn't dropped.

**What you'll see:** Prints that 3 replicas = 3 GPUs, that a 140 GB model's cold start is ~448 s download + 60 s load = 508 s (~8 min) before readiness passes, then the two fixes (cache PVC, warm min replicas) and the `/health` probe note.

## 6 - Autoscale on the right signal

A CPU-utilization HPA never fires on a vLLM pod, because vLLM pre-allocates its GPU and KV-cache VRAM at startup - CPU and GPU-memory stay flat under any load. This block contrasts a CPU HPA against a queue-depth HPA (`vllm:num_requests_waiting`) across three load levels to show which signal actually moves.

In [ ]:
# Autoscale on the RIGHT signal. CPU is useless here: vLLM PRE-ALLOCATES the GPU + KV-cache VRAM at startup,
# so CPU and GPU-memory stay flat no matter the load -> a CPU/memory HPA never scales. Scale on the QUEUE.
CPU_FLAT = 32                    # vLLM sits ~32% CPU whether idle or slammed (VRAM is pre-allocated)
CPU_TARGET = 70                 # a classic CPU HPA target
QUEUE_THRESHOLD = 10            # scale when vllm:num_requests_waiting exceeds this
print("load          CPU%   queue(waiting)   CPU-HPA         queue-HPA (vllm:num_requests_waiting)")
for load, queue in [("light", 0), ("busy", 4), ("overloaded", 38)]:
    cpu_act = "SCALE" if CPU_FLAT > CPU_TARGET else "no-op (never fires)"
    q_act = "SCALE UP" if queue > QUEUE_THRESHOLD else "hold"
    print("  {:<11}   {:>3}       {:>3}           {:<20} {}".format(load, CPU_FLAT, queue, cpu_act, q_act))
print()
print("The CPU HPA is flat at {}% forever, so it NEVER scales - the classic broken LLM autoscaler.".format(CPU_FLAT))
print("Scale on vllm:num_requests_waiting (the queue) or vllm:gpu_cache_usage_perc (the KV cache filling up).")
print("Chain: vLLM exports the metric -> Prometheus -> the k8s custom-metrics API -> HPA (GKE can now do it natively).")

# Output:
# load          CPU%   queue(waiting)   CPU-HPA         queue-HPA (vllm:num_requests_waiting)
#   light          32         0           no-op (never fires)  hold
#   busy           32         4           no-op (never fires)  hold
#   overloaded     32        38           no-op (never fires)  SCALE UP
#
# The CPU HPA is flat at 32% forever, so it NEVER scales - the classic broken LLM autoscaler.
# Scale on vllm:num_requests_waiting (the queue) or vllm:gpu_cache_usage_perc (the KV cache filling up).
# Chain: vLLM exports the metric -> Prometheus -> the k8s custom-metrics API -> HPA (GKE can now do it natively).

**Explanation**

A side-by-side decision table for two autoscaling signals. It holds CPU at a flat constant across every load level (the whole point - it never crosses the HPA target) while the request queue climbs, then prints what each HPA would decide. Read it as proof that a flat metric is a useless autoscaling signal and the queue is the one that responds.

**How the code works - step by step**
- **`CPU_FLAT = 32`** - vLLM's roughly constant CPU whether idle or slammed, because VRAM is pre-allocated.
- **`CPU_TARGET = 70`** - a classic CPU HPA threshold that `CPU_FLAT` never reaches.
- **`QUEUE_THRESHOLD = 10`** - scale when `vllm:num_requests_waiting` exceeds this.
- **loop over `light/busy/overloaded`** - each with a queue depth (0, 4, 38); `cpu_act` is always no-op, `q_act` scales up only when the queue crosses the threshold.
- **Signal note** - scale on `vllm:num_requests_waiting` or `vllm:gpu_cache_usage_perc`.
- **Chain note** - vLLM exports the metric -> Prometheus scrapes -> k8s custom-metrics API -> HPA (GKE can now do this natively).

**In one line:** CPU stays flat and never fires; the queue rises under load, so scale on the queue.

**What you'll see:** Prints a table where CPU is stuck at 32% across light/busy/overloaded (CPU-HPA never fires) while the queue-HPA holds until the overloaded row's queue of 38 trips SCALE UP, plus the metric-chain note.

## 7 - Production economics

Cost per million tokens is simply the GPU price per hour divided by the tokens it serves per hour - so vLLM's throughput is the whole game, because the GPU costs the same per hour whether idle or saturated. This block prices naive versus vLLM serving on the same GPU to show why higher throughput makes each token an order of magnitude cheaper.

In [ ]:
# Cost per 1M tokens = GPU price/hour divided by tokens/hour. vLLM's throughput is what makes self-serving
# cheap: the same GPU that costs the same per hour serves far more tokens, so each token costs far less.
GPU_HR = 3.50                    # illustrative H100 $/hr
for name, tps in [("naive static serving", 200), ("vLLM (continuous + paged)", 2400)]:
    tok_per_hr = tps * 3600
    per_1m = GPU_HR / tok_per_hr * 1_000_000
    print("  {:<28} {:>4} tok/s -> ${:.2f} per 1M tokens".format(name, tps, per_1m))
print()
naive = GPU_HR / (200 * 3600) * 1e6
vllm = GPU_HR / (2400 * 3600) * 1e6
print("Same GPU, same ${:.2f}/hr: vLLM is {:.0f}x cheaper per token purely from higher throughput.".format(GPU_HR, naive / vllm))
print("Stacking optimisations (quantization, chunked prefill, a good prefix-cache hit rate) cuts $/1M another 40-55%.")
print("The decision: vLLM on GKE wins at HIGH, STEADY scale (a busy GPU is cheap per token); below that, a managed")
print("API or Ollama (Lesson 13.5) is simpler and cheaper - self-serving only pays off when the GPUs stay full.")

# Output:
#   naive static serving          200 tok/s -> $4.86 per 1M tokens
#   vLLM (continuous + paged)    2400 tok/s -> $0.41 per 1M tokens
#
# Same GPU, same $3.50/hr: vLLM is 12x cheaper per token purely from higher throughput.
# Stacking optimisations (quantization, chunked prefill, a good prefix-cache hit rate) cuts $/1M another 40-55%.
# The decision: vLLM on GKE wins at HIGH, STEADY scale (a busy GPU is cheap per token); below that, a managed
# API or Ollama (Lesson 13.5) is simpler and cheaper - self-serving only pays off when the GPUs stay full.

**Explanation**

A cost-per-token calculator, the decision model that closes the lesson. It fixes one GPU hourly price and, for two throughput levels, converts tokens/second into tokens/hour and divides the hourly cost across them. Because the numerator (GPU $/hr) is constant, the entire cost difference comes from throughput - the same lever the earlier blocks pull.

**How the code works - step by step**
- **`GPU_HR = 3.50`** - an illustrative H100 dollars-per-hour.
- **loop over naive (200 tok/s) and vLLM (2400 tok/s)** - `tok_per_hr = tps * 3600`, then `per_1m = GPU_HR / tok_per_hr * 1e6`.
- **`naive / vllm`** - the cost ratio between the two, driven purely by throughput.
- **Stacking note** - quantization, chunked prefill, and a good prefix-cache hit rate cut $/1M another 40-55%.
- **Decision** - vLLM on GKE wins at high, steady scale; below that a managed API or Ollama (13.5) is simpler and cheaper.

**In one line:** same GPU $/hr / more tokens/hr = cheaper tokens, so keep the GPU full or rent an API.

**What you'll see:** Prints that naive serving costs $4.86 per 1M tokens versus vLLM's $0.41 on the same $3.50/hr GPU (12x cheaper purely from throughput), plus the 40-55% stacking note and the high-steady-scale decision rule.

## 8 - The GKE deployment, end to end

This final cell ties the whole stack into one illustrative, minimal GKE recipe: a GPU node pool with the modern driver flag, a version-pinned vLLM Deployment on port 8000, a queue-based HPA, and the unchanged OpenAI client your app uses. It is a reference artifact, not something the notebook runs.

In [ ]:
# PRODUCTION vLLM ON GKE - an illustrative minimal example (a GPU node pool, the Deployment, a queue HPA).
# 1) A GPU node pool with the MODERN driver flag (no manual DaemonSet), autoscaling the nodes:
# gcloud container node-pools create gpu-pool --cluster prod --machine-type a3-highgpu-1g \
  # --accelerator type=nvidia-h100-80gb,count=1,gpu-driver-version=default \
  # --enable-autoscaling --min-nodes 1 --max-nodes 8

# 2) The vLLM Deployment - one GPU per pod, OpenAI-compatible on :8000, weights cached on a PVC:
#    image: vllm/vllm-openai:v0.11.0                 # PIN a version, never :latest
#    args: --model meta-llama/Llama-3.3-70B-Instruct-FP8
#          --tensor-parallel-size 1 --gpu-memory-utilization 0.90 --max-model-len 8192 --max-num-seqs 256
#    resources: { limits: { nvidia.com/gpu: 1 } }
#    readinessProbe: { httpGet: { path: /health, port: 8000 }, initialDelaySeconds: 300 }  # cold start

# 3) Autoscale on the QUEUE, not CPU (vLLM pre-allocates VRAM, so CPU never moves):
#    HorizontalPodAutoscaler -> metric vllm:num_requests_waiting, averageValue 10, minReplicas 2

# 4) Your app calls it UNCHANGED - the same OpenAI client, pointed at the in-cluster service:
#    client = OpenAI(base_url="http://vllm-service/v1", api_key="cluster")
# Output: (illustrative minimal example - needs a GKE cluster + GPU quota + the model pulled; not run here.)

**Explanation**

A commented reference manifest, not executable Python - it needs a real GKE cluster, GPU quota, and the model pulled. It collects the concrete flags each earlier block argued for (tensor-parallel-size, gpu-memory-utilization, max-model-len, max-num-seqs, the queue metric, the warm min-replicas floor, the /health probe) into the four steps of a production deployment so you can see how the models map to actual configuration.

**How the code works - step by step**
- **Step 1 - GPU node pool** - `gcloud container node-pools create` with `gpu-driver-version=default` (no manual DaemonSet) and node autoscaling 1-8.
- **Step 2 - vLLM Deployment** - image pinned to `vllm/vllm-openai:v0.11.0` (never `:latest`), the FP8 model, the serving flags, `nvidia.com/gpu: 1`, and a `/health` readiness probe with a 300 s initial delay for the cold start.
- **Step 3 - HPA** - scales on `vllm:num_requests_waiting` (averageValue 10, minReplicas 2), not CPU.
- **Step 4 - the app** - an unchanged OpenAI client pointed at the in-cluster service URL.

**In one line:** node pool -> pinned vLLM Deployment with a queue HPA -> the same OpenAI client, unchanged.

**What you'll see:** No computation - the cell is entirely commented reference config. The only output is the closing note that it is an illustrative minimal example requiring a GKE cluster, GPU quota, and the pulled model, and is not run here.

Seven small keyless models trace one story: continuous batching refills a finished GPU slot from the queue every forward pass so the card never idles, PagedAttention pages the KV cache so far more sequences fit, and together they turn a fixed hourly GPU cost into an order-of-magnitude cheaper token. The `max-num-seqs` knob sets batch fullness, tensor parallelism shards a too-big model across GPUs behind one OpenAI-compatible endpoint, and GKE runs the whole thing as a Deployment designed around the minutes-long cold start and autoscaled on the request queue rather than flat CPU. The final economics model is the decision rule: self-hosting on GKE wins only at high, steady utilization - below that a managed API or Ollama (13.5) is simpler and cheaper. Next stop is Module 14 (LLMOps), where serving becomes an operational discipline of instrumentation, evaluation, and cost optimization.